<a href="https://colab.research.google.com/github/polreig/StartUp_DecoAI/blob/main/Notebook_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalar el ecosistema

In [ ]:
!pip install -q -U google-genai diffusers transformers accelerate opencv-python

## Importar librerías y cargar modelos

In [ ]:
import torch
import cv2
import numpy as np
import json
import requests
from PIL import Image
from io import BytesIO
from pydantic import BaseModel
from typing import Tuple, Dict, Any

from google import genai
from google.genai import types
from google.colab import userdata
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel

print("1. Cargando credenciales de Gemini...")
GOOGLE_API_KEY = userdata.get('clave_API_gemini')
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

print("2. Cargando ControlNet y Stable Diffusion a la GPU...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

print("¡Sistemas listos!")

## Función transformar habitación

In [ ]:
# --- 1. ESQUEMA DE DATOS PARA GEMINI ---
class AnalisisDecoracion(BaseModel):
    estilo_actual: str
    recomendacion: str
    paleta_colores: list[str]
    prompt_sd: str
    muebles_clave: list[str]

# --- 2. FUNCIÓN DE IMAGEN INTELIGENTE ---
def preparar_imagen(ruta_o_url: str, max_size: int = 512) -> Image.Image:
    """Descarga y redimensiona la imagen manteniendo la proporción exacta."""
    try:
        if ruta_o_url.startswith('http'):
            response = requests.get(ruta_o_url, timeout=10)
            response.raise_for_status() # Falla si la URL está rota (ej. error 404)
            img = Image.open(BytesIO(response.content)).convert("RGB")
        else:
            img = Image.open(ruta_o_url).convert("RGB")
            
        # Calcular proporciones para no aplastar los muebles
        ancho, alto = img.size
        if ancho > alto:
            nuevo_ancho = max_size
            nuevo_alto = int((alto / ancho) * max_size)
        else:
            nuevo_alto = max_size
            nuevo_ancho = int((ancho / alto) * max_size)
            
        # Stable Diffusion necesita que los píxeles sean múltiplos de 8
        nuevo_ancho = (nuevo_ancho // 8) * 8
        nuevo_alto = (nuevo_alto // 8) * 8
        
        return img.resize((nuevo_ancho, nuevo_alto), Image.Resampling.LANCZOS)
    except Exception as e:
        raise ValueError(f"Error al cargar la imagen. Revisa la URL o el archivo: {e}")

# --- 3. FUNCIÓN PRINCIPAL DE TRANSFORMACIÓN ---
def transformar_habitacion(ruta_o_url: str, peticion_usuario: str) -> Tuple[Dict[str, Any], Image.Image, Image.Image, Image.Image]:
    print("📥 Cargando y ajustando imagen original...")
    imagen_original = preparar_imagen(ruta_o_url)

    # FASE 1: ANÁLISIS ESTRUCTURADO CON GEMINI
    print("🧠 Analizando la habitación con Gemini (Modo JSON)...")
    prompt_gemini = f"""
    Eres un diseñador de interiores experto. El cliente pide: '{peticion_usuario}'.
    Analiza la foto y extrae los datos solicitados.
    El 'prompt_sd' debe ser una única frase en inglés, muy descriptiva, 
    optimizada para Stable Diffusion (ej: 'modern living room, leather sofa, highly detailed, 8k').
    """
    
    try:
        respuesta_gemini = gemini_client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[imagen_original, prompt_gemini],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=AnalisisDecoracion,
            )
        )
        # Convertimos la respuesta segura en un diccionario de Python
        datos_analisis = json.loads(respuesta_gemini.text)
    except Exception as e:
        raise RuntimeError(f"Error al conectar con Gemini: {e}")

    # FASE 2: EXTRACCIÓN DE ESTRUCTURA (Canny Edge)
    print("📐 Extrayendo plano de la habitación (ControlNet)...")
    imagen_cv = np.array(imagen_original)
    bordes = cv2.Canny(imagen_cv, 100, 200)
    imagen_bordes = Image.fromarray(np.stack([bordes, bordes, bordes], axis=2))

    # FASE 3: GENERACIÓN
    print("🎨 Dibujando el nuevo diseño...")
    prompt_negativo = "low quality, bad anatomy, worst quality, cartoon, illustration, distorted, messy, deformed furniture"

    imagen_generada = pipe(
        datos_analisis["prompt_sd"],
        negative_prompt=prompt_negativo,
        image=imagen_bordes,
        num_inference_steps=25
    ).images[0]

    return datos_analisis, imagen_original, imagen_bordes, imagen_generada

## Prueba

In [ ]:
# Variables de prueba
mi_foto = "https://concept-u.es/cdn/shop/articles/habitacion-comoda.webp?v=1703692446"
#mi_foto = "mi_habitacion.jpg"
mi_peticion = "Quiero convertir esto en una habitación de estilo industrial, con ladrillo visto y muebles de metal negro."

# Ejecutamos el pipeline
analisis, prompt_interno, img_orig, img_bordes, img_final = transformar_habitacion(mi_foto, mi_peticion)

# --- MOSTRAMOS LOS RESULTADOS ---
print("\n" + "="*50)
print(analisis)
print("="*50 + "\n")
print(f"(Prompt enviado a la IA generadora: {prompt_interno})\n")

# Mostramos las imágenes una al lado de la otra usando matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_orig)
axes[0].set_title("1. Original")
axes[0].axis("off")

axes[1].imshow(img_bordes)
axes[1].set_title("2. Estructura (ControlNet)")
axes[1].axis("off")

axes[2].imshow(img_final)
axes[2].set_title("3. Nuevo Diseño")
axes[2].axis("off")

plt.show()

## Función generadora de links

In [ ]:
import urllib.parse
from IPython.display import display, HTML

def generar_links_compra(analisis_cliente):
    print("🛒 Buscando productos recomendados...")

    # 1. Le pedimos a Gemini que extraiga exactamente 3 muebles del texto anterior
    prompt_extraccion = f"""
    Lee este análisis de diseño de interiores:
    '{analisis_cliente}'

    Extrae los 3 muebles o elementos decorativos más importantes que se necesitan comprar para lograr este estilo.
    Responde ÚNICAMENTE con los nombres de los 3 elementos separados por comas.
    Ejemplo: sofá de cuero negro, lámpara de techo industrial, mesa de centro de metal
    """

    respuesta_muebles = gemini_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[prompt_extraccion]
    )

    # 2. Limpiamos el texto y lo convertimos en una lista de Python
    texto_muebles = respuesta_muebles.text.strip()
    lista_muebles = [mueble.strip() for mueble in texto_muebles.split(",")]

    # 3. Generamos los links dinámicos
    html_links = "<h3>🛍️ Lista de la Compra Recomendada:</h3><ul>"

    for mueble in lista_muebles:
        if not mueble: continue # Por si hay algún elemento vacío

        # urllib.parse.quote convierte espacios en %20 o + para que las URLs funcionen bien
        busqueda_codificada = urllib.parse.quote(mueble)

        # Construimos los links reales de búsqueda
        link_ikea = f"https://www.ikea.com/es/es/search/?q={busqueda_codificada}"
        link_amazon = f"https://www.amazon.es/s?k={busqueda_codificada}"

        # Creamos el HTML para que se vea bonito y clickeable en Colab
        html_links += f"""
        <li style='margin-bottom: 10px;'>
            <b>{mueble.capitalize()}</b>:
            <a href='{link_ikea}' target='_blank' style='color: #0051ba; text-decoration: none;'>[ Buscar en IKEA ]</a> |
            <a href='{link_amazon}' target='_blank' style='color: #ff9900; text-decoration: none;'>[ Buscar en Amazon ]</a>
        </li>
        """

    html_links += "</ul>"

    return html_links

## Probamos los links

In [ ]:
# Usamos el 'analisis' que guardamos en la celda anterior
html_resultado = generar_links_compra(analisis)

# En Colab, display(HTML()) renderiza el código HTML como si fuera una web
display(HTML(html_resultado))